In [33]:
import sys
sys.path.append('.')
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wfdb
import neurokit2 as nk
import warnings
from scipy.signal import resample
from sklearn.metrics import roc_auc_score, average_precision_score
import math

# -------------------------
# Config
# -------------------------
BEAT_LENGTH  = 300
MAX_BEATS    = 35
IN_CHANNELS  = 12
BEAT_DIM     = IN_CHANNELS * BEAT_LENGTH

D_MODEL      = 256
NUM_HEADS    = 8
NUM_LAYERS   = 4
MLP_RATIO    = 4
DROPOUT      = 0.1

In [34]:
class HREncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

    def forward(self, rr):
        # rr: (B, N)
        return self.mlp(rr.unsqueeze(-1))   # (B, N, d_model)


class ResampleCNNWithHRTokenizer(nn.Module):
    def __init__(self, in_channels=12, d_model=256, beat_length=300):
        super().__init__()
        self.beat_length = beat_length
        self.proj = nn.Conv1d(
            in_channels=in_channels,
            out_channels=d_model,
            kernel_size=beat_length,
            stride=beat_length,
            bias=True,
        )
        self.hr_encoder = HREncoder(d_model)

    def forward(self, beats, rr_intervals):
        B, N, C, T = beats.shape
        x = beats.view(B * N, C, T)
        z = self.proj(x)                    # (B*N, d_model, 1)
        z = z.squeeze(-1)                   # (B*N, d_model)
        cnn_embeddings = z.view(B, N, -1)   # (B, N, d_model)
        hr_embeddings  = self.hr_encoder(rr_intervals)   # (B, N, d_model)
        return cnn_embeddings + hr_embeddings


class SinCosPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        assert d_model % 2 == 0
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(dtype=x.dtype, device=x.device)


class TransformerEncoder(nn.Module):
    def __init__(self, d_model=256, num_heads=8, num_layers=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * mlp_ratio,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x, src_key_padding_mask=None):
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)


class ECGMaskedSSLBeatHR(nn.Module):
    def __init__(self, in_channels=12, d_model=256, beat_length=300,
                 num_heads=8, num_layers=4, mlp_ratio=4, dropout=0.1, max_beats=35):
        super().__init__()
        self.beat_dim   = in_channels * beat_length
        self.tokenizer  = ResampleCNNWithHRTokenizer(in_channels, d_model, beat_length)
        self.posenc     = SinCosPositionalEncoding(max_beats, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.encoder    = TransformerEncoder(d_model, num_heads, num_layers, mlp_ratio, dropout)
        self.pred_head  = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, self.beat_dim),
        )

    def forward(self, beats, rr_intervals, padding_mask=None, mask=None,
                mask_ratio=0.50, span_len=1):
        tokens  = self.tokenizer(beats, rr_intervals)   # (B, N, d_model)
        B, N, D = tokens.shape

        target_patches = beats.reshape(B, N, self.beat_dim)

        if mask is None:
            mask = torch.zeros(B, N, dtype=torch.bool, device=tokens.device)
        if padding_mask is not None:
            mask = mask & ~padding_mask

        mask_token    = self.mask_token.expand(B, N, D)
        masked_tokens = torch.where(mask.unsqueeze(-1), mask_token, tokens)
        masked_tokens = self.posenc(masked_tokens)
        encoded       = self.encoder(masked_tokens, src_key_padding_mask=padding_mask)
        pred_patches  = self.pred_head(encoded)

        if padding_mask is not None:
            real   = (~padding_mask).unsqueeze(-1).float()
            pooled = (encoded * real).sum(dim=1) / real.sum(dim=1).clamp(min=1)
        else:
            pooled = encoded.mean(dim=1)

        return {
            "pred_patches":   pred_patches,
            "target_patches": target_patches,
            "mask":           mask,
            "encoded":        encoded,
            "pooled":         pooled,
        }

In [35]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

checkpoint_path = Path("checkpoints_tok3_ssl/best.pt")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

pretrained_model = ECGMaskedSSLBeatHR(
    in_channels=IN_CHANNELS,
    d_model=D_MODEL,
    beat_length=BEAT_LENGTH,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    mlp_ratio=MLP_RATIO,
    dropout=DROPOUT,
    max_beats=MAX_BEATS,
).to(device)

state_dict = checkpoint["model_state_dict"]
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
pretrained_model.load_state_dict(state_dict)
pretrained_model.eval()
print("Loaded checkpoint from:", checkpoint_path)

Loaded checkpoint from: checkpoints_tok3_ssl/best.pt


/tmp/ipykernel_2246106/1691685856.py:68: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [36]:
class PTBXLBeatHRClassifier(nn.Module):
    def __init__(self, pretrained_model, feature_dim=256, num_classes=1):
        super().__init__()
        self.pretrained_model = pretrained_model
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, beats, rr_intervals, padding_mask):
        out    = self.pretrained_model(beats, rr_intervals, padding_mask=padding_mask)
        pooled = out["pooled"]
        return self.classifier(pooled)


model = PTBXLBeatHRClassifier(pretrained_model, num_classes=5).to(device)
print(model)

PTBXLBeatHRClassifier(
  (pretrained_model): ECGMaskedSSLBeatHR(
    (tokenizer): ResampleCNNWithHRTokenizer(
      (proj): Conv1d(12, 256, kernel_size=(300,), stride=(300,))
      (hr_encoder): HREncoder(
        (mlp): Sequential(
          (0): Linear(in_features=1, out_features=64, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=64, out_features=256, bias=True)
        )
      )
    )
    (posenc): SinCosPositionalEncoding()
    (encoder): TransformerEncoder(
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0-3): 4 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
            )
            (linear1): Linear(in_features=256, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=1024, out_features=256, bias=True)
            

In [37]:
ptbxl_base = Path("/data/rohit/PTB-XL")
label_df   = pd.read_csv(ptbxl_base / "ptbxl_database.csv")
scp_df     = pd.read_csv(ptbxl_base / "scp_statements.csv", index_col=0)

label_df["scp_codes"] = label_df["scp_codes"].apply(ast.literal_eval)

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

scp_to_superclass = {}
for scp_code, row in scp_df.iterrows():
    if row["diagnostic_class"] in SUPERCLASSES:
        scp_to_superclass[scp_code] = row["diagnostic_class"]

def get_superclass_labels(scp_codes_dict):
    labels = np.zeros(len(SUPERCLASSES), dtype=np.float32)
    for scp_code in scp_codes_dict.keys():
        if scp_code in scp_to_superclass:
            superclass = scp_to_superclass[scp_code]
            idx = SUPERCLASSES.index(superclass)
            labels[idx] = 1.0
    return labels

label_df["target"] = label_df["scp_codes"].apply(get_superclass_labels)
label_df = label_df[label_df["target"].apply(lambda x: x.sum() > 0)].copy()

label_df["ecg_path"] = label_df.apply(lambda row: ptbxl_base / row["filename_hr"], axis=1)

def files_exist(row):
    base = Path(row["ecg_path"])
    return Path(str(base) + ".hea").exists() and Path(str(base) + ".dat").exists()

label_df["file_exists"] = label_df.apply(files_exist, axis=1)
label_df = label_df[label_df["file_exists"]].copy()

train_df = label_df[label_df["strat_fold"] <= 8].copy()
val_df   = label_df[label_df["strat_fold"] == 9].copy()
test_df  = label_df[label_df["strat_fold"] == 10].copy()

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
for i, sc in enumerate(SUPERCLASSES):
    n = label_df["target"].apply(lambda x: x[i]).sum()
    print(f"  {sc}: {int(n)}")

Train: 17111
Val:   2156
Test:  2163
  NORM: 9528
  MI: 5486
  STTC: 5250
  CD: 4907
  HYP: 2655


In [38]:
def detect_r_peaks(lead_ii, sampling_rate=500):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _, info = nk.ecg_peaks(lead_ii, sampling_rate=sampling_rate)
        return np.array(info["ECG_R_Peaks"])
    except Exception:
        return np.array([])


def extract_beats_with_rr(x, beat_length=300, sampling_rate=500, min_beats=3):
    """
    x: (12, 5000) z-scored
    returns:
        beat_array:  (N, 12, beat_length) resampled
        rr_intervals:(N,) R-R interval in seconds
    or None, None if detection fails
    """
    lead_ii = x[1]
    r_peaks = detect_r_peaks(lead_ii, sampling_rate=sampling_rate)

    if len(r_peaks) < 2:
        return None, None

    beats      = []
    rr_seconds = []

    for i in range(len(r_peaks) - 1):
        start = r_peaks[i]
        end   = r_peaks[i + 1]
        beat  = x[:, start:end]
        if beat.shape[1] < 10:
            continue

        rr = (end - start) / sampling_rate
        rr_seconds.append(rr)

        resampled = np.zeros((12, beat_length), dtype=np.float32)
        for c in range(12):
            resampled[c] = resample(beat[c], beat_length)
        beats.append(resampled)

    if len(beats) < min_beats:
        return None, None

    return np.stack(beats, axis=0), np.array(rr_seconds, dtype=np.float32)


class PTBXLBeatHRDataset(Dataset):
    def __init__(self, df, beat_length=300):
        self.df          = df.reset_index(drop=True)
        self.beat_length = beat_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        record_path = str(row["ecg_path"])

        record = wfdb.rdrecord(record_path)
        x      = record.p_signal.astype(np.float32).T   # (12, 5000)

        x = np.clip(x, -5, 5)
        mean = x.mean(axis=1, keepdims=True)
        std  = x.std(axis=1, keepdims=True)
        x    = (x - mean) / np.clip(std, 1e-4, None)

        beats, rr_intervals = extract_beats_with_rr(x, beat_length=self.beat_length)
        if beats is None:
            return self.__getitem__((idx + 1) % len(self))

        y = torch.tensor(row["target"], dtype=torch.float32)   # (5,)
        return torch.from_numpy(beats), torch.from_numpy(rr_intervals), y


def beat_hr_collate_fn(batch):
    beats_list, rr_list, labels = zip(*batch)
    num_beats = [b.shape[0] for b in beats_list]
    max_n     = max(num_beats)
    B         = len(beats_list)

    padded_beats = torch.zeros(B, max_n, IN_CHANNELS, BEAT_LENGTH)
    padded_rr    = torch.zeros(B, max_n)
    padding_mask = torch.ones(B, max_n, dtype=torch.bool)

    for i, (beats, rr) in enumerate(zip(beats_list, rr_list)):
        n = beats.shape[0]
        padded_beats[i, :n] = beats
        padded_rr[i, :n]    = rr
        padding_mask[i, :n] = False

    return padded_beats, padded_rr, padding_mask, torch.stack(labels)

In [39]:
train_dataset = PTBXLBeatHRDataset(train_df, beat_length=BEAT_LENGTH)
val_dataset   = PTBXLBeatHRDataset(val_df,   beat_length=BEAT_LENGTH)
test_dataset  = PTBXLBeatHRDataset(test_df,  beat_length=BEAT_LENGTH)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=8, collate_fn=beat_hr_collate_fn,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=8, collate_fn=beat_hr_collate_fn,
                          persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=8, collate_fn=beat_hr_collate_fn,
                          persistent_workers=True, prefetch_factor=2)

print("Dataloaders ready")

Dataloaders ready


In [40]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for beats, rr_intervals, padding_mask, y in loader:
        beats        = beats.to(device)
        rr_intervals = rr_intervals.to(device)
        padding_mask = padding_mask.to(device)
        y            = y.to(device)   # (B, 5)

        optimizer.zero_grad()
        logits = model(beats, rr_intervals, padding_mask)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for beats, rr_intervals, padding_mask, y in loader:
            beats        = beats.to(device)
            rr_intervals = rr_intervals.to(device)
            padding_mask = padding_mask.to(device)
            y            = y.to(device)   # (B, 5)

            logits = model(beats, rr_intervals, padding_mask)
            loss   = criterion(logits, y)
            probs  = torch.sigmoid(logits)

            total_loss  += loss.item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    aucs   = []
    auprcs = []
    class_aucs   = {}
    class_auprcs = {}

    for i, sc in enumerate(SUPERCLASSES):
        if all_labels[:, i].sum() > 0:
            auc   = roc_auc_score(all_labels[:, i], all_probs[:, i])
            auprc = average_precision_score(all_labels[:, i], all_probs[:, i])
            aucs.append(auc)
            auprcs.append(auprc)
            class_aucs[sc]   = auc
            class_auprcs[sc] = auprc

    macro_auc   = np.mean(aucs)
    macro_auprc = np.mean(auprcs)

    return total_loss / len(loader), macro_auc, macro_auprc, class_aucs, class_auprcs


num_epochs   = 20
best_val_auprc = 0.0

downstream_ckpt_dir = Path("checkpoints_tok3_downstream")
downstream_ckpt_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_auc, val_auprc, val_class_aucs, val_class_auprcs = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss     : {train_loss:.4f}")
    print(f"  Val Loss       : {val_loss:.4f}")
    print(f"  Val Macro AUC  : {val_auc:.4f}")
    print(f"  Val Macro AUPRC: {val_auprc:.4f}")
    for sc in SUPERCLASSES:
        if sc in val_class_aucs:
            print(f"    {sc}: AUC={val_class_aucs[sc]:.4f}  AUPRC={val_class_auprcs[sc]:.4f}")
    print("-" * 40)

    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        torch.save(model.state_dict(), downstream_ckpt_dir / "best.pt")
        print(f"  Saved new best model (val Macro AUPRC: {val_auprc:.4f})")

best_state = torch.load(downstream_ckpt_dir / "best.pt", map_location=device)
model.load_state_dict(best_state)
print(f"\nLoaded best model (val Macro AUPRC: {best_val_auprc:.4f})")

test_loss, test_auc, test_auprc, test_class_aucs, test_class_auprcs = evaluate(
    model, test_loader, criterion, device
)

print("\n========== TEST RESULTS ==========")
print(f"Test Loss       : {round(test_loss, 4)}")
print(f"Test Macro AUC  : {round(test_auc, 4)}")
print(f"Test Macro AUPRC: {round(test_auprc, 4)}")
print("\nPer-class results:")
for sc in SUPERCLASSES:
    if sc in test_class_aucs:
        print(f"  {sc}: AUC={round(test_class_aucs[sc], 4)}  AUPRC={round(test_class_auprcs[sc], 4)}")

Epoch 1/20
  Train Loss     : 0.3741
  Val Loss       : 0.3491
  Val Macro AUC  : 0.8691
  Val Macro AUPRC: 0.6784
    NORM: AUC=0.9118  AUPRC=0.8735
    MI: AUC=0.8720  AUPRC=0.7103
    STTC: AUC=0.9051  AUPRC=0.7180
    CD: AUC=0.8742  AUPRC=0.7475
    HYP: AUC=0.7823  AUPRC=0.3428
----------------------------------------
  Saved new best model (val Macro AUPRC: 0.6784)
Epoch 2/20
  Train Loss     : 0.3244
  Val Loss       : 0.3345
  Val Macro AUC  : 0.8803
  Val Macro AUPRC: 0.7072
    NORM: AUC=0.9201  AUPRC=0.8856
    MI: AUC=0.8842  AUPRC=0.7363
    STTC: AUC=0.9159  AUPRC=0.7514
    CD: AUC=0.8898  AUPRC=0.7797
    HYP: AUC=0.7916  AUPRC=0.3832
----------------------------------------
  Saved new best model (val Macro AUPRC: 0.7072)
Epoch 3/20
  Train Loss     : 0.3090
  Val Loss       : 0.3370
  Val Macro AUC  : 0.8793
  Val Macro AUPRC: 0.7099
    NORM: AUC=0.9134  AUPRC=0.8737
    MI: AUC=0.8885  AUPRC=0.7453
    STTC: AUC=0.9141  AUPRC=0.7483
    CD: AUC=0.8961  AUPRC=0.7965